In [71]:
import pandas as pd
import psycopg

In [72]:
conn = psycopg.connect(
    "dbname=dailyedge_development"
)

start_date = "2025-09-01"
end_date = "2026-07-07"

rth_start = "08:30"
rth_end = "15:15"

In [73]:
query = """
SELECT timestamp, open, high, low, close
FROM CANDLES
WHERE timestamp >= %s
  AND timestamp < %s::date + INTERVAL '1 day'
  AND timestamp::time BETWEEN %s AND %s
ORDER BY timestamp;
"""

df = pd.read_sql(
    query,
    conn,
    params=(start_date, end_date, rth_start, rth_end)
)

df["timestamp"] = pd.to_datetime(df["timestamp"])

/tmp/ipykernel_55078/1413650733.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


In [74]:
df["date"] = df["timestamp"].dt.date

sessions = {
    date: session.reset_index(drop=True)
    for date, session in df.groupby("date")
}

In [75]:
def get_target_direction(session, target_distance):
    open_price = session.iloc[0]["open"]

    long_target = open_price + target_distance
    short_target = open_price - target_distance

    for _, candle in session.iterrows():
        hit_long = candle["high"] >= long_target
        hit_short = candle["low"] <= short_target

        if hit_long and hit_short:
            return "Ambiguous"

        if hit_long:
            return "Long"

        if hit_short:
            return "Short"

    return "Neither"

In [76]:
def evaluate_continuous_trail_long(session, trail_distance, target_distance):
    open_price = session.iloc[0]["open"]

    target_price = open_price + target_distance
    stop_price = open_price - trail_distance

    for _, candle in session.iterrows():
        # The stop entering this candle is fixed for the entire candle.
        active_stop = stop_price

        # Check the candle open first.
        if candle["open"] <= active_stop:
            return active_stop - open_price

        hit_stop = candle["low"] <= active_stop
        hit_target = candle["high"] >= target_price

        # Both occurred inside the same candle.
        if hit_stop and hit_target:
            return "Unknown"

        if hit_stop:
            return active_stop - open_price

        if hit_target:
            return target_distance

        # Update once, after this candle, for the next candle.
        stop_price = max(
            active_stop,
            candle["high"] - trail_distance
        )

    return session.iloc[-1]["close"] - open_price

In [77]:
def evaluate_continuous_trail_short(session, trail_distance, target_distance):
    open_price = session.iloc[0]["open"]

    target_price = open_price - target_distance
    stop_price = open_price + trail_distance

    for _, candle in session.iterrows():
        # The stop entering this candle is fixed for the entire candle.
        active_stop = stop_price

        # Check the candle open first.
        if candle["open"] >= active_stop:
            return open_price - active_stop

        hit_stop = candle["high"] >= active_stop
        hit_target = candle["low"] <= target_price

        # Both occurred inside the same candle.
        if hit_stop and hit_target:
            return "Unknown"

        if hit_stop:
            return open_price - active_stop

        if hit_target:
            return target_distance

        # Update once, after this candle, for the next candle.
        stop_price = min(
            active_stop,
            candle["low"] + trail_distance
        )

    return open_price - session.iloc[-1]["close"]

In [78]:
def evaluate_continuous_trade(session, trail_distance, target_distance):
    direction = get_target_direction(session, target_distance)

    if direction == "Long":
        pnl = evaluate_continuous_trail_long(
            session,
            trail_distance,
            target_distance
        )

    elif direction == "Short":
        pnl = evaluate_continuous_trail_short(
            session,
            trail_distance,
            target_distance
        )

    else:
        pnl = None

    return {
        "direction": direction,
        "pnl": pnl
    }

In [79]:
results_50_100 = []

for date, session in sessions.items():
    result = evaluate_continuous_trade(
        session,
        trail_distance=50,
        target_distance=100
    )

    results_50_100.append({
        "date": date,
        "direction": result["direction"],
        "pnl": result["pnl"]
    })

results_50_100 = pd.DataFrame(results_50_100)

In [80]:
resolved_50_100 = results_50_100[
    pd.to_numeric(results_50_100["pnl"], errors="coerce").notna()
].copy()

resolved_50_100["pnl"] = resolved_50_100["pnl"].astype(float)

In [81]:
average_loss = abs(
    resolved_50_100.loc[
        resolved_50_100["pnl"] < 0,
        "pnl"
    ].mean()
)

full_win = 100
effective_rr = full_win / average_loss

print("Average actual loss:", average_loss)
print("Full win:", full_win)
print("Effective R:R: 1 :", effective_rr)

Average actual loss: 28.017140481481256
Full win: 100
Effective R:R: 1 : 3.5692436230634565


In [82]:
parameter_sets = [
    (5, 10),
    (15, 25),
    (25, 50),
    (35, 70),
    (50, 70),
    (50, 100),
    (75, 100),
    (75, 150),
]

In [83]:
def evaluate_parameter_set(trail_distance, target_distance):
    results = []

    for date, session in sessions.items():
        result = evaluate_continuous_trade(
            session,
            trail_distance=trail_distance,
            target_distance=target_distance
        )

        results.append({
            "date": date,
            "direction": result["direction"],
            "pnl": result["pnl"]
        })

    return pd.DataFrame(results)

In [84]:
def summarize_parameter_set(trail_distance, target_distance):
    results = evaluate_parameter_set(
        trail_distance,
        target_distance
    )

    numeric_pnl = pd.to_numeric(results["pnl"], errors="coerce")
    resolved = results[numeric_pnl.notna()].copy()
    resolved["pnl"] = numeric_pnl[numeric_pnl.notna()]

    full_wins = resolved["pnl"] == target_distance
    stopped = resolved["pnl"] < target_distance
    losses = resolved["pnl"] < 0

    average_loss = abs(resolved.loc[losses, "pnl"].mean())

    return {
        "trail": trail_distance,
        "target": target_distance,
        "resolved": len(resolved),
        "full_wins": full_wins.sum(),
        "stopped": stopped.sum(),
        "actual_losses": losses.sum(),
        "avg_stopped_pnl": resolved.loc[stopped, "pnl"].mean(),
        "avg_actual_loss": average_loss,
        "expectancy": resolved["pnl"].mean(),
        "effective_rr": target_distance / average_loss,
    }

In [85]:
summary_rows = [
    summarize_parameter_set(trail, target)
    for trail, target in parameter_sets
]

summary = pd.DataFrame(summary_rows)

summary

,trail,target,resolved,full_wins,stopped,actual_losses,avg_stopped_pnl,avg_actual_loss,expectancy,effective_rr
0,5,10,87,87,0,0,NaN,NaN,10.000000,NaN
1,15,25,163,127,36,26,-8.375056,13.758129,17.628822,1.817108
2,25,50,205,97,108,57,-4.161290,18.584534,21.466247,2.690409
3,35,70,211,80,131,84,-7.316726,21.196839,21.997672,3.302379
4,50,70,212,120,92,73,-19.138053,26.857977,31.317449,2.606302
5,50,100,206,61,145,81,-4.607092,28.017140,26.368794,3.569244
6,75,100,206,113,93,67,-24.216396,37.721562,43.921724,2.651004
7,75,150,179,61,118,66,-9.611561,42.283129,44.781206,3.547514


In [86]:
def get_add_on_price(open_price, direction, trail_distance):
    if direction == "Long":
        return open_price + trail_distance

    if direction == "Short":
        return open_price - trail_distance

In [87]:
def evaluate_continuous_trail_add_on_long(
    session,
    trail_distance,
    target_distance
):
    open_price = session.iloc[0]["open"]

    target_price = open_price + target_distance
    add_on_price = get_add_on_price(
        open_price,
        "Long",
        trail_distance
    )

    stop_price = open_price - trail_distance
    added = False

    for _, candle in session.iterrows():
        active_stop = stop_price

        # Existing stop is active at the candle open.
        if candle["open"] <= active_stop:
            pnl_1 = active_stop - open_price

            if added:
                pnl_2 = active_stop - add_on_price
                return pnl_1 + pnl_2

            return pnl_1

        hit_stop = candle["low"] <= active_stop
        hit_target = candle["high"] >= target_price

        if hit_stop and hit_target:
            return "Unknown"

        if hit_stop:
            pnl_1 = active_stop - open_price

            if added:
                pnl_2 = active_stop - add_on_price
                return pnl_1 + pnl_2

            return pnl_1

        if hit_target:
            pnl_1 = target_distance

            # Reaching the target necessarily crosses the add-on price first.
            if added or target_price >= add_on_price:
                pnl_2 = target_price - add_on_price
                return pnl_1 + pnl_2

            return pnl_1

        # Add the second unit once the threshold has been reached.
        if not added and candle["high"] >= add_on_price:
            added = True

        # Trail updates once for the next candle.
        stop_price = max(
            active_stop,
            candle["high"] - trail_distance
        )

    eod_price = session.iloc[-1]["close"]

    pnl_1 = eod_price - open_price

    if added:
        pnl_2 = eod_price - add_on_price
        return pnl_1 + pnl_2

    return pnl_1

In [88]:
def evaluate_continuous_trail_add_on_short(
    session,
    trail_distance,
    target_distance
):
    open_price = session.iloc[0]["open"]

    target_price = open_price - target_distance
    add_on_price = get_add_on_price(
        open_price,
        "Short",
        trail_distance
    )

    stop_price = open_price + trail_distance
    added = False

    for _, candle in session.iterrows():
        active_stop = stop_price

        # Existing stop is active at the candle open.
        if candle["open"] >= active_stop:
            pnl_1 = open_price - active_stop

            if added:
                pnl_2 = add_on_price - active_stop
                return pnl_1 + pnl_2

            return pnl_1

        hit_stop = candle["high"] >= active_stop
        hit_target = candle["low"] <= target_price

        if hit_stop and hit_target:
            return "Unknown"

        if hit_stop:
            pnl_1 = open_price - active_stop

            if added:
                pnl_2 = add_on_price - active_stop
                return pnl_1 + pnl_2

            return pnl_1

        if hit_target:
            pnl_1 = target_distance

            # Reaching the target necessarily crosses the add-on price first.
            if added or target_price <= add_on_price:
                pnl_2 = add_on_price - target_price
                return pnl_1 + pnl_2

            return pnl_1

        # Add the second unit once the threshold has been reached.
        if not added and candle["low"] <= add_on_price:
            added = True

        # Trail updates once for the next candle.
        stop_price = min(
            active_stop,
            candle["low"] + trail_distance
        )

    eod_price = session.iloc[-1]["close"]

    pnl_1 = open_price - eod_price

    if added:
        pnl_2 = add_on_price - eod_price
        return pnl_1 + pnl_2

    return pnl_1

In [89]:
def evaluate_continuous_add_on_trade(
    session,
    trail_distance,
    target_distance
):
    direction = get_target_direction(session, target_distance)

    if direction == "Long":
        pnl = evaluate_continuous_trail_add_on_long(
            session,
            trail_distance,
            target_distance
        )

    elif direction == "Short":
        pnl = evaluate_continuous_trail_add_on_short(
            session,
            trail_distance,
            target_distance
        )

    else:
        pnl = None

    return {
        "direction": direction,
        "pnl": pnl
    }

In [90]:
def summarize_add_on_parameter_set(trail_distance, target_distance):
    results = []

    for date, session in sessions.items():
        result = evaluate_continuous_add_on_trade(
            session,
            trail_distance,
            target_distance
        )

        results.append(result["pnl"])

    pnl = pd.to_numeric(pd.Series(results), errors="coerce")
    resolved = pnl.dropna()

    full_win_pnl = (
        target_distance
        + (target_distance - trail_distance)
    )

    full_wins = resolved == full_win_pnl
    losses = resolved < 0

    average_loss = abs(resolved[losses].mean())

    return {
        "trail": trail_distance,
        "target": target_distance,
        "resolved": len(resolved),
        "full_wins": full_wins.sum(),
        "actual_losses": losses.sum(),
        "avg_actual_loss": average_loss,
        "expectancy": resolved.mean(),
        "full_win_pnl": full_win_pnl,
        "effective_rr": full_win_pnl / average_loss,
    }

In [91]:
add_on_summary_rows = [
    summarize_add_on_parameter_set(trail, target)
    for trail, target in parameter_sets
]

add_on_summary = pd.DataFrame(add_on_summary_rows)

add_on_summary

,trail,target,resolved,full_wins,actual_losses,avg_actual_loss,expectancy,full_win_pnl,effective_rr
0,5,10,87,87,0,NaN,15.000000,15,NaN
1,15,25,163,127,33,12.286119,24.844830,35,2.848743
2,25,50,205,97,83,16.845371,30.051121,75,4.452262
3,35,70,211,80,108,20.609977,31.367546,105,5.094620
4,50,70,212,120,92,27.290972,39.100144,90,3.297794
5,50,100,206,61,112,27.262812,33.414232,150,5.502000
6,75,100,206,113,93,42.224786,49.505315,125,2.960347
7,75,150,179,61,99,40.406030,57.806470,225,5.568476


In [92]:
def evaluate_continuous_add_on_direction(
    session,
    direction,
    trail_distance,
    target_distance
):
    if direction == "Long":
        return evaluate_continuous_trail_add_on_long(
            session,
            trail_distance,
            target_distance
        )

    if direction == "Short":
        return evaluate_continuous_trail_add_on_short(
            session,
            trail_distance,
            target_distance
        )

    return None

In [93]:
accuracy_results = []

for trail_distance, target_distance in parameter_sets:
    for date, session in sessions.items():
        actual_direction = get_target_direction(
            session,
            target_distance
        )

        if actual_direction not in ["Long", "Short"]:
            continue

        wrong_direction = (
            "Short" if actual_direction == "Long" else "Long"
        )

        correct_pnl = evaluate_continuous_add_on_direction(
            session,
            actual_direction,
            trail_distance,
            target_distance
        )

        wrong_pnl = evaluate_continuous_add_on_direction(
            session,
            wrong_direction,
            trail_distance,
            target_distance
        )

        accuracy_results.append({
            "trail": trail_distance,
            "target": target_distance,
            "date": date,
            "actual_direction": actual_direction,
            "correct_pnl": correct_pnl,
            "wrong_pnl": wrong_pnl
        })

accuracy_results = pd.DataFrame(accuracy_results)

In [94]:
side_summary_rows = []

for trail_distance, target_distance in parameter_sets:
    subset = accuracy_results[
        (accuracy_results["trail"] == trail_distance) &
        (accuracy_results["target"] == target_distance)
    ].copy()

    correct = pd.to_numeric(subset["correct_pnl"], errors="coerce")
    wrong = pd.to_numeric(subset["wrong_pnl"], errors="coerce")

    side_summary_rows.append({
        "trail": trail_distance,
        "target": target_distance,
        "avg_correct_pnl": correct.mean(),
        "avg_wrong_pnl": wrong.mean(),
        "wrong_side_resolved": wrong.notna().sum()
    })

side_summary = pd.DataFrame(side_summary_rows)

side_summary

,trail,target,avg_correct_pnl,avg_wrong_pnl,wrong_side_resolved
0,5,10,15.000000,-5.000000,121
1,15,25,24.844830,-13.932377,202
2,25,50,30.051121,-19.137577,214
3,35,70,31.367546,-22.656594,213
4,50,70,39.100144,-34.137022,213
5,50,100,33.414232,-30.455364,208
6,75,100,49.505315,-50.221732,208
7,75,150,57.806470,-49.062274,179


In [95]:
accuracy_levels = [
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
]

expectancy_rows = []

for _, row in side_summary.iterrows():
    result = {
        "Stop / Target": f"{int(row['trail'])} / {int(row['target'])}"
    }

    for accuracy in accuracy_levels:
        expectancy = (
            accuracy * row["avg_correct_pnl"]
            + (1 - accuracy) * row["avg_wrong_pnl"]
        )

        result[f"{int(accuracy * 100)}%"] = expectancy

    expectancy_rows.append(result)

expectancy_by_accuracy = pd.DataFrame(expectancy_rows)

expectancy_by_accuracy

,Stop / Target,50%,55%,60%,65%,70%,75%,80%
0,5 / 10,5.000000,6.000000,7.000000,8.000000,9.000000,10.000000,11.000000
1,15 / 25,5.456226,7.395087,9.333947,11.272807,13.211668,15.150528,17.089389
2,25 / 50,5.456772,7.916206,10.375641,12.835076,15.294511,17.753946,20.213381
3,35 / 70,4.355476,7.056683,9.757890,12.459097,15.160304,17.861511,20.562718
4,50 / 70,2.481561,6.143419,9.805277,13.467136,17.128994,20.790852,24.452711
5,50 / 100,1.479434,4.672914,7.866393,11.059873,14.253353,17.446833,20.640312
6,75 / 100,-0.358208,4.628144,9.614496,14.600849,19.587201,24.573553,29.559906
7,75 / 150,4.372098,9.715535,15.058972,20.402410,25.745847,31.089284,36.432721


In [96]:
single_accuracy_results = []

for trail_distance, target_distance in parameter_sets:
    for date, session in sessions.items():
        actual_direction = get_target_direction(
            session,
            target_distance
        )

        if actual_direction not in ["Long", "Short"]:
            continue

        wrong_direction = (
            "Short" if actual_direction == "Long" else "Long"
        )

        if actual_direction == "Long":
            correct_pnl = evaluate_continuous_trail_long(
                session,
                trail_distance,
                target_distance
            )
            wrong_pnl = evaluate_continuous_trail_short(
                session,
                trail_distance,
                target_distance
            )
        else:
            correct_pnl = evaluate_continuous_trail_short(
                session,
                trail_distance,
                target_distance
            )
            wrong_pnl = evaluate_continuous_trail_long(
                session,
                trail_distance,
                target_distance
            )

        single_accuracy_results.append({
            "trail": trail_distance,
            "target": target_distance,
            "date": date,
            "correct_pnl": correct_pnl,
            "wrong_pnl": wrong_pnl
        })

single_accuracy_results = pd.DataFrame(single_accuracy_results)

In [97]:
single_expectancy_rows = []

for trail_distance, target_distance in parameter_sets:
    subset = single_accuracy_results[
        (single_accuracy_results["trail"] == trail_distance) &
        (single_accuracy_results["target"] == target_distance)
    ]

    correct = pd.to_numeric(
        subset["correct_pnl"],
        errors="coerce"
    )

    wrong = pd.to_numeric(
        subset["wrong_pnl"],
        errors="coerce"
    )

    avg_correct = correct.mean()
    avg_wrong = wrong.mean()

    result = {
        "Stop / Target": f"{trail_distance} / {target_distance}"
    }

    for accuracy in accuracy_levels:
        expectancy = (
            accuracy * avg_correct
            + (1 - accuracy) * avg_wrong
        )

        result[f"{int(accuracy * 100)}%"] = expectancy

    single_expectancy_rows.append(result)

single_expectancy_by_accuracy = pd.DataFrame(
    single_expectancy_rows
)

single_expectancy_by_accuracy

,Stop / Target,50%,55%,60%,65%,70%,75%,80%
0,5 / 10,2.500000,3.250000,4.000000,4.750000,5.500000,6.250000,7.000000
1,15 / 25,2.111508,3.663239,5.214971,6.766702,8.318433,9.870165,11.421896
2,25 / 50,2.055208,3.996312,5.937416,7.878520,9.819624,11.760728,13.701832
3,35 / 70,1.742172,3.767722,5.793272,7.818822,9.844372,11.869922,13.895472
4,50 / 70,1.436822,4.424885,7.412947,10.401010,13.389073,16.377135,19.365198
5,50 / 100,0.719305,3.284254,5.849203,8.414152,10.979101,13.544050,16.108999
6,75 / 100,-0.018677,4.375364,8.769404,13.163444,17.557484,21.951524,26.345564
7,75 / 150,2.341903,6.585834,10.829764,15.073694,19.317624,23.561555,27.805485
